In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import tensorflow as tf
import numpy as np

from deep_lss.models.delta_model import DeltaLossModel
from deep_lss.nets.resnet import ResNetLayers
from msfm.utils import files

23-08-29 02:25:07    scales.py INF   Setting up healpy to run on 256 CPUs 


The goal of this notebook is to fix the following error:

```
Cannot use GPU when output.shape[1] * nnz(a) > 2^31 [Op:SparseTensorDenseMatMul]

Call arguments received by layer "chebyshev" (type Chebyshev):
  • input_tensor=tf.Tensor(shape=(208, 7264, 128), dtype=float32)
  • training=False
```

In [3]:
n_side = 512
data_vec_pix, _, _, _ = files.load_pixel_file()
n_neighbors = 20

# data
n_params = 6
n_batch = 16
n_batch *= (2 * n_params + 1)
n_pix = len(data_vec_pix)
n_z = 8
batch = tf.random.uniform((n_batch, n_pix, n_z), seed=10)
print(batch.shape)

23-08-29 02:25:07     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_512.h5 
(208, 464896, 8)


In [4]:
# network = ResNetLayers(
#     n_base_channels=32,
#     # depth
#     n_downsampling=3,
#     n_cheby=2,
#     n_residuals=5,
#     # misc
#     poly_degree=5,   
# ).get_layers()

network = ResNetLayers(
    n_base_channels=32,
    # depth
    n_downsampling=2,
    n_cheby=1,
    n_residuals=3,
    # misc
    poly_degree=5,   
).get_layers()

In [5]:
# build the model
model = DeltaLossModel(
    network=network,
    n_side=n_side,
    indices=data_vec_pix,
    n_neighbors=n_neighbors,
    input_shape=(None, n_pix, n_z),
    max_batch_size=n_batch,
    checkpoint_dir=None,
    summary_dir=None,
    restore_checkpoint=False,
)

23-08-29 02:25:08 delta_model. INF   Initializing DeltaLossModel with a HealpyGCNN model 
Detected a reduction factor of 8.0, the input with nside 512 will be transformed to 64 during a forward pass. Checking for consistency with indices...
indices seem consistent...
Tracing... Due to tensor size, tf.sparse.sparse_dense_matmul is executed over 4 splits. Beware of the resulting performance penalty.
Tracing... Due to tensor size, tf.sparse.sparse_dense_matmul is executed over 2 splits. Beware of the resulting performance penalty.
Model: "healpy_gcnn_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 healpy_pseudo_conv (HealpyP  (None, 116224, 32)       1056      
 seudoConv)                                                      
                                                                 
 healpy_pseudo_conv_1 (Healp  (None, 29056, 64)        8256      
 yPseudoConv)                                  

In [6]:
preds = model(batch)

In [7]:
# def extended_sparse_dense_matmul(sparse_tensor, dense_tensor, n_splits=1):
#     """
#     Splits axis 1 of the dense_tensor such that tensorflow can handle the size of the computation.
#     :param sparse_tensor: Input sparse tensor of rank 2. 
#     :param dense_tensor: Input dense tensor of rank 2.
#     :param n_splits: Integer number of splits applied to axis 1 of dense_tensor.

#     For reference, the error message to be avoided:

#     'Cannot use GPU when output.shape[1] * nnz(a) > 2^31 [Op:SparseTensorDenseMatMul]

#     Call arguments received by layer "chebyshev" (type Chebyshev):
#     • input_tensor=tf.Tensor(shape=(208, 7264, 128), dtype=float32)
#     • training=False'
#     """
#     if n_splits > 1:
#         dense_splits = tf.split(dense_tensor, n_splits, axis=1)
#         result = []
#         for dense_split in dense_splits:
#             result.append(tf.sparse.sparse_dense_matmul(sparse_tensor, dense_split))
#         result = tf.concat(result, axis=1)
#     else:
#         result = tf.sparse.sparse_dense_matmul(sparse_tensor, dense_tensor)

#     return result

# tf_extended_sparse_dense_matmul = tf.function(extended_sparse_dense_matmul)

In [8]:
from deepsphere.utils import split_sparse_dense_matmul

In [9]:
np.random.seed(12)
L_size = 7264
n_samples = 1000

L = tf.sparse.SparseTensor(
    indices=np.random.choice(np.arange(L_size), 2*n_samples).reshape(-1, 2), 
    values=np.asarray(np.random.uniform(size=n_samples), dtype=np.float32), 
    dense_shape=(L_size, L_size)
)

x0 = tf.random.uniform(shape=(L_size, 6656))

no_split = split_sparse_dense_matmul(L, x0)
split = split_sparse_dense_matmul(L, x0, 8)

print(np.all(np.isclose(no_split, split)))

Tracing... Due to tensor size, tf.sparse.sparse_dense_matmul is executed over 8 splits. Beware of the resulting performance penalty.
True


In [10]:
%%timeit
split_sparse_dense_matmul(L, x0)

403 µs ± 22.4 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [11]:
%%timeit
split_sparse_dense_matmul(L, x0, 8)

1.16 ms ± 7.19 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
